<a href="https://colab.research.google.com/github/Prahladh-Vulsa/Skill-RAG-Nexus/blob/main/Skill_RAG_Nexus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [ ]:
# Install all required packages
!pip install -qU langchain-google-genai
!pip install -qU langchain langchain-tavily
!pip install -qU langchain-mcp-adapters
!pip install -qU langchain-community pypdf
!pip install -qU langchain-text-splitter
!pip install -qU langchain-huggingface sentence_transformers
!pip install -qU langchain-chroma

Imports and Environment Setup

In [ ]:
import os
import requests
from google.colab import userdata
from sentence_transformers import CrossEncoder
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.agents import create_agent

In [ ]:
# 1. Load API keys cleanly
google_api_key = userdata.get('gemini_api_key')
tavily_api_key = userdata.get('tavily_apikey')
rapid_api_key = userdata.get('rapid_apikey')

# 2. Setup Gemini LLM
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

# 3. Setup Embedding and Reranker models
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 4. Text splitter configuration
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

Simple PDF Ingestion

In [ ]:
def load_pdf_to_vectorstore(pdf_path, collection_name, persist_dir):
    """Loads a PDF if it exists, otherwise creates an empty collection cleanly."""
    store = Chroma(
        collection_name=collection_name,
        embedding_function=embeddings,
        persist_directory=persist_dir
    )

    # Simple check: does the file exist?
    if os.path.exists(pdf_path):
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        splits = text_splitter.split_documents(docs)
        store.add_documents(splits)
        print(f"Loaded {len(splits)} chunks from {pdf_path}")
    else:
        print(f"Notice: '{pdf_path}' not found. Created empty store.")

    return store

# Process PDFs cleanly
research_vector_store = load_pdf_to_vectorstore(
    "/content/attention_is_all_you_need.pdf",
    "research_collection",
    "./chroma_research_db"
)

resume_vector_store = load_pdf_to_vectorstore(
    "/content/RESUME.pdf",
    "resume_collection",
    "./chroma_resume_db"
)

Custom Tools

In [ ]:
def rerank(query: str, docs: list, top_n: int = 3) -> list:
    """Helper to pick the top 3 best matching chunks using the Cross-Encoder."""
    if not docs:
        return []

    # Pair query with text, score them, and sort
    pairs = [[query, d.page_content] for d in docs]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in ranked[:top_n]]


@tool
def retrieve_from_research_pdf(query: str) -> str:
    """Retrieve precise academic facts from the research paper PDF."""
    docs = research_vector_store.similarity_search(query, k=10)
    top_docs = rerank(query, docs, top_n=3)

    if not top_docs:
        return "No relevant facts found in the research paper."

    return "\n\n".join([f"Content: {d.page_content}" for d in top_docs])


@tool
def retrieve_from_resume(query: str) -> str:
    """Retrieve skills and projects from the uploaded resume PDF."""
    docs = resume_vector_store.similarity_search(query, k=10)
    top_docs = rerank(query, docs, top_n=3)

    if not top_docs:
        return "No relevant details found in the resume."

    return "\n\n".join([f"Content: {d.page_content}" for d in top_docs])


@tool
def search_jobs(skill: str, location: str) -> str:
    """Search live job listings requiring specific skills via RapidAPI."""
    if not rapid_api_key:
        return "RapidAPI key is missing."

    url = "https://jsearch.p.rapidapi.com/search"
    headers = {"x-rapidapi-key": rapid_api_key, "x-rapidapi-host": "jsearch.p.rapidapi.com"}
    params = {"query": f"{skill} in {location}", "page": "1", "country": "in"}

    response = requests.get(url, headers=headers, params=params)

    # Simple check on HTTP status
    if response.status_code != 200:
        return "Failed to fetch jobs from API."

    jobs = response.json().get("data", [])
    if not jobs:
        return f"No jobs found for {skill} in {location}."

    results = []
    for job in jobs[:3]:
        results.append(f"- {job.get('job_title')} at {job.get('employer_name')} ({job.get('job_apply_link')})")

    return "\n".join(results)

System Prompt

In [ ]:
system_prompt = """You are a multi-purpose assistant capable of executing technical research, career counseling, and job matching.

You have access to these specific tools:
1. mcp_tavily: Search web trends or broad updates.
2. search_jobs: Search live job listings based on skills and locations.
3. retrieve_from_research_pdf: Query facts from the research paper.
4. retrieve_from_resume: Query skills and experience from the resume.

Routing Strategy:
- Use retrieve_from_research_pdf for paper/concept questions.
- Use retrieve_from_resume to extract candidate skills first, then use search_jobs to find openings.
- Present job results cleanly in plain text with application links.
"""

Agent Runner

In [ ]:
async def run_unified_agent(user_query: str):
    # Fetch MCP Tavily tools dynamically
    client = MultiServerMCPClient({
        "mcp_tavily": {
            "transport": "http",
            "url": f"https://mcp.tavily.com/mcp/?tavilyApiKey={tavily_api_key}",
        }
    })
    mcp_tools = await client.get_tools()

    # Combine all tools
    all_tools = mcp_tools + [search_jobs, retrieve_from_research_pdf, retrieve_from_resume]

    # Create and execute the agent
    agent = create_agent(
        model=model,
        tools=all_tools,
        system_prompt=system_prompt,
        debug=True
    )

    response = await agent.ainvoke({
        "messages": [{"role": "user", "content": user_query}]
    })

    print("\n--- AGENT RESPONSE ---")
    print(response["messages"][-1].content[0]["text"])

Run Query

In [ ]:
query = """
1. What are the main concepts explained in the research paper?
2. Check my resume to see if I have experience related to those concepts.
3. Find 2 matching live job openings in Bangalore.
"""

await run_unified_agent(query)